# KOSPI Index Prediction — Linear, Ridge, Lasso and Polynomial Regression

**Day 1 · Regression Analysis lab** (English version of *코스피지수_예측.ipynb*)

We predict the KOSPI index level from 29 macro-financial indicators: 160 rows (May 2003 – Aug 2016), each treated as an independent observation and split at random, 80/20.

**The lab:** OLS → Ridge → Lasso → polynomial features → regularized polynomial models, compared side by side in section 8.

**Changes from the Korean original**
1. Data file with English column names and a `Date` column (`KOSPI_Index_EN.csv`, UTF-8).
2. `r2_score` is called as `r2_score(y_true, y_pred)`. The original `r2(y_pred, y_test)` swapped the arguments; R² is **not** symmetric.

## 0. Setup

In [1]:
import os
FILE = 'KOSPI_Index_EN.csv'

# In Colab: upload KOSPI_Index_EN.csv when prompted (or drag it into the Files panel)
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print('Please upload', FILE)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'{FILE} not found in {os.getcwd()}')

In [2]:
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 12)

## 1. Load and inspect the data

In [3]:
kospi = pd.read_csv(FILE)
kospi.shape

(160, 31)

In [4]:
kospi.head()

,Date,FKI BSI,Housing Price Index,Construction BSI (Outlook),Unemployment Rate (SA %),Dishonored Bill Rate,...,Gross Investment Rate,CRB Commodity Futures,CRB Energy,CRB Industrials,CRB Precious Metals,KOSPI
0,2003-05,96.4,70.13,83.3,3.7,0.09,...,32.3,243.7,247.7,217.5,322.4,633.4
1,2003-06,90.3,70.83,86.7,3.7,0.08,...,32.3,247.6,255.7,247.3,326.4,669.9
2,2003-07,91.4,70.57,86.7,3.8,0.06,...,31.8,249.9,268.7,232.3,340.1,713.5
3,2003-08,109.6,69.99,92.4,3.9,0.06,...,31.8,255.3,283.3,256.6,364.1,759.5
4,2003-09,110.3,69.70,64.1,3.8,0.08,...,31.8,262.6,292.7,262.0,368.3,697.5


Column groups (29 features + target):

| Group | Columns |
|---|---|
| Sentiment | FKI BSI, Construction BSI (Outlook), OECD Leading Indicator (Korea) |
| Real economy & credit | Housing Price Index, Unemployment Rate (SA %), Dishonored Bill Rate, Outbound Travelers, Outbound Travelers YoY |
| Trade | Daily Exports YoY |
| Rates & spreads | MSB 364D Yield, Public Corp Bond AAA 3Y, Bank Bond AAA 3Y, KTB 5Y - CD, KTB 10Y - MSB 1Y, Bank AAA 3Y - KTB 3Y, Corp AA- 3Y - KTB 3Y |
| Market risk | VKOSPI |
| National accounts (quarterly, repeated monthly) | GDP Services, Equipment Investment, Construction Investment, Goods Exports, Domestic Demand, GDP Deflator, Gross Saving Rate, Gross Investment Rate |
| Commodities | CRB Commodity Futures, CRB Energy, CRB Industrials, CRB Precious Metals |
| **Target** | **KOSPI** (month-end close) |

*The `Date` column was not in the source file; it was inferred by matching KOSPI month-end closes (e.g., 2008-10 = 1,113.1; 2016-08 = 2,034.7). See `KOSPI_Data_Dictionary.csv` for the Korean original names.*

In [5]:
X = kospi.drop(columns=['Date', 'KOSPI'])   # features
y = kospi['KOSPI']                           # target
dates = pd.to_datetime(kospi['Date'])
print(X.shape, y.shape)
y.head()

(160, 29) (160,)


0    633.4
1    669.9
2    713.5
3    759.5
4    697.5
Name: KOSPI, dtype: float64

In [6]:
y.tail()

155    1994.2
156    1983.4
157    1970.4
158    2016.2
159    2034.7
Name: KOSPI, dtype: float64

# The lab

## 2. Train / test split

In [7]:
from sklearn.model_selection import train_test_split

# Same setting as the original notebook: random 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
columns = X_train.columns
X_train.head()

,FKI BSI,Housing Price Index,Construction BSI (Outlook),Unemployment Rate (SA %),Dishonored Bill Rate,Daily Exports YoY,...,Gross Saving Rate,Gross Investment Rate,CRB Commodity Futures,CRB Energy,CRB Industrials,CRB Precious Metals
60,95.3,85.62,61.6,3.3,0.02,19.08,...,32.9,34.2,452.4,873.6,378.3,646.1
115,85.7,93.31,70.8,3.1,0.02,1.62,...,34.1,28.6,541.8,879.6,481.4,1088.3
2,91.4,70.57,86.7,3.8,0.06,14.83,...,33.3,31.8,249.9,268.7,232.3,340.1
123,94.4,93.80,66.5,3.1,0.02,8.15,...,34.4,28.9,508.1,924.0,489.2,916.1
45,112.3,80.94,92.2,3.3,0.02,15.62,...,33.0,33.0,410.4,661.9,430.8,633.1


### Standardize the features
Fit the scaler on the **training set only**, then apply the same transformation to the test set.

In [8]:
from sklearn.preprocessing import StandardScaler

std = StandardScaler()
X_train = std.fit_transform(X_train)
X_test = std.transform(X_test)   # do NOT fit here: that would leak test information

## 3. Linear regression

In [9]:
# 1. Instantiate
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

In [10]:
# 2. Fit
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [11]:
# 3. Predict
y_pred = lr.predict(X_test)

In [12]:
# 4. Evaluate: RMSE and R^2
from sklearn.metrics import mean_squared_error as mse, r2_score as r2

print('RMSE: ', math.sqrt(mse(y_test, y_pred)))
print('R^2 : ', r2(y_test, y_pred) * 100)   # r2(y_true, y_pred): order matters!

RMSE:  107.51042716353047
R^2 :  94.6046933492504


In [13]:
# RMSE relative to the average test-set level
math.sqrt(mse(y_test, y_pred)) / np.mean(y_test)

np.float64(0.06763456634981717)

In [14]:
# Coefficients (standardized features), sorted by size
coef_lr = pd.Series(lr.coef_, index=columns)
coef_lr.sort_values(key=abs, ascending=False).round(1).head(10)

Bank Bond AAA 3Y           852.4
Public Corp Bond AAA 3Y   -728.9
Housing Price Index        307.0
CRB Commodity Futures      203.9
CRB Precious Metals        -90.3
Outbound Travelers          70.7
KTB 5Y - CD                -59.1
Corp AA- 3Y - KTB 3Y       -55.1
Bank AAA 3Y - KTB 3Y       -48.8
MSB 364D Yield             -41.7
dtype: float64

> Notice **Public Corp Bond AAA 3Y ≈ −729** and **Bank Bond AAA 3Y ≈ +852**. The two yields are almost identical (corr 0.997), so OLS assigns them huge offsetting weights. Read the pair together, not one by one — this is multicollinearity.

## 4. Ridge regression

In [15]:
# 1. Instantiate
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=100)

# 2. Fit
ridge_model.fit(X_train, y_train)

# 3. Predict
y_pred = ridge_model.predict(X_test)

# 4. Evaluate: RMSE and R^2
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)

# Coefficients
pd.Series(ridge_model.coef_, index=columns).round(1).to_frame('ridge_coef').T

RMSE: 159.2749923751821
R^2 : 88.15840660486015


,FKI BSI,Housing Price Index,Construction BSI (Outlook),Unemployment Rate (SA %),Dishonored Bill Rate,Daily Exports YoY,...,Gross Saving Rate,Gross Investment Rate,CRB Commodity Futures,CRB Energy,CRB Industrials,CRB Precious Metals
ridge_coef,8.1,82.4,-2.6,-26.9,-51.7,-14.5,...,8.3,-3.6,53.8,37.7,36.3,46.0


## 5. Lasso regression

In [16]:
# 1. Instantiate
from sklearn.linear_model import Lasso
lasso_model = Lasso(alpha=100)

# 2. Fit
lasso_model.fit(X_train, y_train)

# 3. Predict
y_pred = lasso_model.predict(X_test)

# 4. Evaluate: RMSE and R^2
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)

# Coefficients: Lasso sets most of them exactly to zero
coef_lasso = pd.Series(lasso_model.coef_, index=columns)
coef_lasso[coef_lasso != 0].round(1)

RMSE: 173.4861535027301
R^2 : 85.95102645191307


Housing Price Index      222.4
Dishonored Bill Rate     -22.7
CRB Commodity Futures     85.8
dtype: float64

## 6. Polynomial regression

In [17]:
from sklearn.preprocessing import PolynomialFeatures

# Instantiate the transformer (degree 2: squares + pairwise interactions + bias column)
poly = PolynomialFeatures(degree=2)

# Transform X: fit_transform = fit + transform on the training set
poly_X_train = poly.fit_transform(X_train)
poly_X_test = poly.transform(X_test)   # the test set gets the same transformation
print(poly_X_train.shape, poly_X_test.shape)   # 465 columns but only 128 training rows

(128, 465) (32, 465)


In [18]:
lr = LinearRegression()
lr.fit(poly_X_train, y_train)
y_pred = lr.predict(poly_X_test)
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred))

RMSE: 129.08140380765073
R^2 : 0.9222245883465182


## 7. The remedy? Lasso and Ridge on polynomial features

In [19]:
lasso_model = Lasso(alpha=100)
lasso_model.fit(poly_X_train, y_train)
y_pred = lasso_model.predict(poly_X_test)
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)
print('non-zero coefficients:', np.sum(lasso_model.coef_ != 0), 'of', lasso_model.coef_.size)

RMSE: 169.74520028255964
R^2 : 86.55038153025177
non-zero coefficients: 7 of 465


In [20]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(poly_X_train, y_train)
y_pred = ridge_model.predict(poly_X_test)
print('RMSE:', math.sqrt(mse(y_test, y_pred)))
print('R^2 :', r2(y_test, y_pred) * 100)

RMSE: 124.69766581398255
R^2 : 92.7417552790614


## 8. Summary (random split)

In [21]:
def evaluate(models, Xtr, Xte, ytr, yte, Ptr=None, Pte=None):
    rows = []
    for name, model, use_poly in models:
        a, b = (Ptr, Pte) if use_poly else (Xtr, Xte)
        model.fit(a, ytr)
        yp = model.predict(b)
        rows.append({'model': name,
                     'test RMSE': math.sqrt(mse(yte, yp)),
                     'test R2': r2(yte, yp),
                     'train R2': r2(ytr, model.predict(a)),
                     'non-zero coefs': int(np.sum(np.abs(model.coef_) > 1e-9))})
    return pd.DataFrame(rows).set_index('model').round(3)

MODELS = lambda: [('Linear regression', LinearRegression(), False),
                  ('Ridge a=100', Ridge(alpha=100), False),
                  ('Lasso a=100', Lasso(alpha=100), False),
                  ('Poly-2 + OLS', LinearRegression(), True),
                  ('Poly-2 + Lasso a=100', Lasso(alpha=100), True),
                  ('Poly-2 + Ridge a=1', Ridge(alpha=1), True)]

res_random = evaluate(MODELS(), X_train, X_test, y_train, y_test, poly_X_train, poly_X_test)
res_random

,test RMSE,test R2,train R2,non-zero coefs
model,,,,
Linear regression,107.510,0.946,0.968,29
Ridge a=100,159.275,0.882,0.909,29
Lasso a=100,173.486,0.860,0.835,3
Poly-2 + OLS,129.081,0.922,1.000,464
Poly-2 + Lasso a=100,169.745,0.866,0.850,7
Poly-2 + Ridge a=1,124.698,0.927,1.000,464


Every model scores a test R² above 0.85 on this random split, and plain OLS has the lowest test RMSE.